<table style="width:100%">
<tr>
<td style="vertical-align:middle; text-align:left;">
<font size="2">
<a href="https://mng.bz/lZ5B">Build a Reasoning Model (From Scratch)</a> 一书的补充代码，作者 <a href="https://sebastianraschka.com">Sebastian Raschka</a><br>
<br>代码仓库：<a href="https://github.com/rasbt/reasoning-from-scratch">https://github.com/rasbt/reasoning-from-scratch</a>
</font>
</td>
<td style="vertical-align:middle; text-align:left;">
<a href="https://mng.bz/lZ5B"><img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/cover-small.webp" width="100px"></a>
</td>
</tr>
</table>

# 附录 D：使用更大的 LLM

本笔记本使用的软件包：

In [1]:
from importlib.metadata import version

used_libraries = [
    "reasoning_from_scratch",  # for download functions
    "torch",
    "tokenizers"
]

for lib in used_libraries:
    print(f"{lib} version: {version(lib)}")

reasoning_from_scratch version: 0.1.17
torch version: 2.10.0
tokenizers version: 0.21.4


- 主要章节使用 Qwen3 0.6B 基础模型，因为它是 Qwen3 系列中最小的模型，因此最容易在消费级硬件上运行
- 然而，附录 C 中相同的 `Qwen3Model` 实现也可以用于加载更大的 Qwen3 密集检查点，使用相同的从零开始的 PyTorch 模型代码

&nbsp;
## D.1 更大的 Qwen3 密集模型配置

本仓库在 `reasoning_from_scratch.appendix_c`（[reasoning_from_scratch/appendix_c.py](https://github.com/rasbt/reasoning-from-scratch/blob/main/reasoning_from_scratch/appendix_c.py)）中包含了几个更大的 Qwen3 密集模型（超出 0.6B 模型）的配置字典：

| 模型大小 | 配置字典 |
| --- | --- |
| 1.7B | `QWEN3_CONFIG_1_7B` |
| 4B | `QWEN3_CONFIG_4B` |
| 8B | `QWEN3_CONFIG_8B` |
| 14B | `QWEN3_CONFIG_14B` |
| 32B | `QWEN3_CONFIG_32B` |

<img src="https://sebastianraschka.com/images/reasoning-from-scratch-images/appendix-d/Appendix_D_F01_raschka.webp" width="500px">

- 如上图所示，这些是“密集” Qwen3 变体，可以在单个 GPU 上运行
- 还有 Qwen3 的“稀疏”混合专家（Mixture-of-Experts）变体，但本书代码不支持它们；不过如果你感兴趣，可以在这里找到一个从零开始的实现：https://github.com/rasbt/LLMs-from-scratch/tree/main/ch05/11_qwen3
- 所有这些都使用与附录 C 中 0.6B 模型相同的整体架构模式
- 变化的是嵌入大小、层数、注意力头数量和前馈隐藏层维度

- 作为粗略的下限估计，以 bfloat16 存储权重每个参数大约需要 2 字节
- 这意味着仅检查点权重就大约在以下量级：

| 模型大小 | bfloat16 下的大致权重内存 |
| --- | --- |
| 1.7B | 约 3.4 GB |
| 4B | 约 8 GB |
| 8B | 约 16 GB |
| 14B | 约 28 GB |
| 32B | 约 64 GB |

- 实际上，真实的运行时内存使用量更高，因为我们还需要用于激活值、临时缓冲区以及通常的 KV 缓存的内存

&nbsp;
## D.2 下载更大检查点概述

- 与主要章节中使用的 0.6B 检查点不同，更大的官方 Qwen3 模型通常以 `safetensors` 文件分发，有时会分成多个分片
- 用于加载这些的辅助函数 `download_from_huggingface_from_snapshots` 需要一些额外的包：

```bash
!uv add huggingface_hub safetensors
```

or

```bash
!pip install huggingface_hub safetensors
```

&nbsp;
## D.3 加载更大的基础模型

- 下载权重：

In [2]:
from pathlib import Path
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.appendix_c import (
    download_from_huggingface_from_snapshots
)


device = get_device()
local_dir = Path("qwen3-4b-base")

weights = download_from_huggingface_from_snapshots(
    repo_id="Qwen/Qwen3-4B-Base",
    local_dir=local_dir,
)

Using Apple Silicon GPU (MPS)


/Users/sebastian/Developer/reasoning-from-scratch/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 13 files: 100%|██████████████████████| 13/13 [00:00<00:00, 2616.79it/s]


- 初始化模型：

In [3]:
from reasoning_from_scratch.qwen3 import (
    Qwen3Model, load_hf_weights_into_qwen
)
from reasoning_from_scratch.appendix_c import QWEN3_CONFIG_4B


model = Qwen3Model(QWEN3_CONFIG_4B)
load_hf_weights_into_qwen(
    model,
    param_config={
        "n_layers": QWEN3_CONFIG_4B["n_layers"],
        "hidden_dim": QWEN3_CONFIG_4B["hidden_dim"],
    },
    params=weights,
)
model.to(device)
model.eval()

Model uses weight tying.


Qwen3Model(
  (tok_emb): Embedding(151936, 2560)
  (trf_blocks): ModuleList(
    (0-35): 36 x TransformerBlock(
      (att): GroupedQueryAttention(
        (W_query): Linear(in_features=2560, out_features=4096, bias=False)
        (W_key): Linear(in_features=2560, out_features=1024, bias=False)
        (W_value): Linear(in_features=2560, out_features=1024, bias=False)
        (out_proj): Linear(in_features=4096, out_features=2560, bias=False)
        (q_norm): RMSNorm()
        (k_norm): RMSNorm()
      )
      (ff): FeedForward(
        (fc1): Linear(in_features=2560, out_features=9728, bias=False)
        (fc2): Linear(in_features=2560, out_features=9728, bias=False)
        (fc3): Linear(in_features=9728, out_features=2560, bias=False)
      )
      (norm1): RMSNorm()
      (norm2): RMSNorm()
    )
  )
  (final_norm): RMSNorm()
  (out_head): Linear(in_features=2560, out_features=151936, bias=False)
)

- 加载分词器：

In [4]:
from reasoning_from_scratch.qwen3 import Qwen3Tokenizer
import shutil

# 注意原始基础分词器名为 "tokenizer.json"
# 我们将其重命名以区分推理分词器（下一节）
tokenizer_src = local_dir / "tokenizer.json"
tokenizer_path = local_dir / "tokenizer-base.json"

if not tokenizer_path.exists():
    shutil.copyfile(tokenizer_src, tokenizer_path)

tokenizer = Qwen3Tokenizer(tokenizer_file_path=tokenizer_path)

- 使用模型：

In [5]:
import torch
from reasoning_from_scratch.ch02 import (
    generate_text_basic_stream_cache,
)

prompt = "Explain large language models in two sentences."
input_ids = torch.tensor(
    tokenizer.encode(prompt),
    device=device,
).unsqueeze(0)

for token in generate_text_basic_stream_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=64,
    eos_token_id=tokenizer.eos_token_id,
):
    print(tokenizer.decode(token.squeeze(0).tolist()), end="", flush=True)

 Large language models are artificial intelligence systems that use deep learning techniques to understand and generate human-like text. They are trained on vast amounts of data and can perform a wide range of natural language processing tasks, such as translation, summarization, and question answering.

&nbsp;
## D.4 加载更大的推理变体

- 同样的方法也适用于更大的推理风格 Qwen3 模型
- 对于给定的模型大小，架构保持不变；只有检查点和分词器设置改变

例如，要加载 4B 推理变体而不是 4B 基础变体，我们需要：

- 将仓库 ID 从 `Qwen/Qwen3-4B-Base` 切换到 `Qwen/Qwen3-4B`；
- 将 `tokenizer.json` 文件复制为 `tokenizer-reasoning.json`；
- 按如下方式初始化分词器：

```python
tokenizer = Qwen3Tokenizer(
    tokenizer_file_path=tokenizer_path,
    apply_chat_template=True,
    add_generation_prompt=True,
    add_thinking=True,
)
```

- 其余的模型加载和使用代码保持不变

&nbsp;
## D.5 实用建议

- 本节没有代码